<a href="https://colab.research.google.com/github/sdantasdaniel-stack/agent-rag-lgpd/blob/main/PROJETO_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJETO FINAL
# DANIEL SILVEIRA DANTAS


In [ ]:
# Instalando dependências necessárias

!pip install langchain langchain-community langchain-groq faiss-cpu pypdf python-dotenv

##  Escolha do Documento

Para este projeto, foi escolhida a **Lei Geral de Proteção de Dados Pessoais (LGPD) — Lei nº 13.709/2018**,
obtida em formato PDF pela edição do Senado Federal (atualizada até junho de 2024).

A LGPD foi escolhida por ser um documento juridicamente relevante, com escopo bem delimitado
(uma única lei), linguagem consistente e estrutura organizada em capítulos e artigos —
características ideais para um sistema RAG. Além disso, o tema é de grande utilidade prática,
pois qualquer pessoa ou empresa que lida com dados pessoais precisa compreender seus direitos
e obrigações segundo a lei.

O formato PDF foi preferido ao HTML do site do Planalto por oferecer um texto mais limpo,
sem ruídos de navegação, facilitando a extração e o processamento do conteúdo.

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

# Upload do PDF
uploaded = files.upload()  # selecione o lgpd.pdf quando abrir a janela

# Carrega o PDF
loader = PyPDFLoader("lgpd.pdf")
pages = loader.load()

print(f"Total de páginas carregadas: {len(pages)}")
print("\n--- Prévia da página 1 ---")
print(pages[0].page_content[:500])

Saving lgpd.pdf to lgpd.pdf
Total de páginas carregadas: 49

--- Prévia da página 1 ---
Lei Geral de Proteção 
de Dados Pessoais
Lei no 13.709/2018
Edição atualizada até junho de 2024


##  Splitting do Documento

Para dividir o documento em chunks, foi utilizado o **RecursiveCharacterTextSplitter**
da biblioteca `langchain_text_splitters`, com os seguintes parâmetros:

- `chunk_size=1000`: cada chunk contém até 1000 caracteres, tamanho suficiente para
  capturar um artigo completo ou uma seção relevante da lei.
- `chunk_overlap=200`: sobreposição de 200 caracteres entre chunks consecutivos,
  garantindo que informações importantes não sejam cortadas abruptamente entre dois pedaços.

O `RecursiveCharacterTextSplitter` foi escolhido por tentar preservar a estrutura natural
do texto, dividindo preferencialmente em parágrafos e frases antes de cortar no meio de
uma sentença. O resultado foram **117 chunks**, cobrindo toda a extensão da lei.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

print(f"Total de chunks: {len(chunks)}")
print("\n--- Prévia do chunk 1 ---")
print(chunks[0].page_content)

Total de chunks: 117

--- Prévia do chunk 1 ---
Lei Geral de Proteção 
de Dados Pessoais
Lei no 13.709/2018
Edição atualizada até junho de 2024


## Criação de Vector Store

Os chunks gerados foram transformados em vetores numéricos (embeddings) utilizando o modelo
**`sentence-transformers/all-MiniLM-L6-v2`**, da biblioteca `HuggingFaceEmbeddings`.
Este modelo foi escolhido por ser leve, gratuito e eficiente para textos em língua portuguesa,
capturando bem a similaridade semântica entre trechos jurídicos.

Os vetores foram armazenados no **FAISS** (Facebook AI Similarity Search), uma biblioteca
otimizada para busca por similaridade em grandes coleções de vetores. O FAISS foi preferido
por rodar localmente sem necessidade de servidor externo, sendo ideal para o ambiente do
Google Colab. Ao final, **117 vetores** foram indexados — um para cada chunk do documento.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Modelo de embeddings gratuito, roda direto no Colab
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Cria o vector store
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector store criado com sucesso! ✅")
print(f"Total de vetores indexados: {vectorstore.index.ntotal}")

/tmp/ipython-input-1862726358.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.war

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store criado com sucesso! ✅
Total de vetores indexados: 117


## Retrieval

O retriever foi configurado com o parâmetro `k=4`, ou seja, a cada consulta o sistema
recupera os 4 chunks mais semanticamente similares à pergunta do usuário.

Este valor foi escolhido como equilíbrio entre contexto suficiente para o LLM formular
uma boa resposta e economia de tokens no prompt. Valores muito baixos (k=1 ou k=2)
podem perder informações relevantes espalhadas em artigos diferentes; valores muito altos
aumentam o custo e podem introduzir ruído.

O retriever utiliza busca por similaridade de cosseno via FAISS, comparando o vetor
da pergunta com os vetores dos chunks para retornar os trechos mais relevantes da LGPD.

## Geração de Respostas

Para a geração de respostas, foi utilizado o modelo **`llama-3.3-70b-versatile`**
via API da **Groq**, integrado ao LangChain através da biblioteca `langchain_groq`.

O Groq foi escolhido por oferecer inferência extremamente rápida e por ser o único encontrado com acesso gratuito
a modelos de grande porte. O `llama-3.3-70b-versatile` é o modelo mais capaz
disponível na plataforma, com excelente desempenho em tarefas de compreensão e
síntese de textos jurídicos.

O parâmetro `temperature=0` foi definido para garantir respostas determinísticas
e precisas — essencial em contexto jurídico, onde respostas criativas ou aleatórias
seriam inadequadas.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools.retriever import create_retriever_tool
from google.colab import userdata

# Carrega a API key com segurança
groq_api_key = userdata.get("GROQ_API_KEY")

# Configura o LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0
)

# Cria o retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Cria a ferramenta RAG
tool = create_retriever_tool(
    retriever,
    name="consultar_lgpd",
    description="Consulta a Lei Geral de Proteção de Dados (LGPD). Use esta ferramenta para responder qualquer pergunta sobre a LGPD."
)

print("LLM e ferramenta RAG configurados com sucesso! ")

LLM e ferramenta RAG configurados com sucesso! 


## 📝 Prompt do Agent

O prompt do agent foi cuidadosamente elaborado para restringir o comportamento do LLM
ao escopo da LGPD, evitando respostas baseadas em conhecimento geral do modelo.

### Decisões de design do prompt

- **Persona jurídica**: o agent é instruído a se comportar como um assistente jurídico
  especializado, promovendo respostas precisas e com linguagem adequada ao contexto legal.
- **Uso da ferramenta**: a instrução de usar a ferramenta `consultar_lgpd` garante que
  toda resposta seja fundamentada no documento original, não no conhecimento genérico do LLM.
- **Citação de artigos**: o agent é orientado a citar o artigo correspondente sempre
  que possível (ex: "conforme o Art. 5º da LGPD..."), aumentando a rastreabilidade
  e confiabilidade das respostas.
- **Limitação de escopo**: se a informação não constar na lei, o agent deve informar
  explicitamente, evitando alucinações — comportamento confirmado nos testes, onde o
  agent respondeu corretamente que a LGPD não menciona "o país mais rico do mundo".

### Problema identificado: loop infinito

Na primeira versão do agent, foi observado um comportamento indesejado: o agent chamava
a ferramenta `consultar_lgpd` repetidamente com a mesma query (mais de 6 vezes seguidas)
sem nunca formular uma resposta final. Isso ocorreu porque o LLM não recebia uma instrução
clara de que deveria encerrar o raciocínio após obter o resultado da ferramenta.

### Solução aplicada

O problema foi resolvido com duas medidas complementares:

1. **Instrução explícita no prompt**: foi adicionada a diretriz *"Use a ferramenta
   consultar_lgpd UMA ÚNICA VEZ e, após receber o resultado, formule sua resposta
   final imediatamente"*, orientando o LLM a não repetir chamadas desnecessárias.

2. **Parâmetro `max_iterations=3` no AgentExecutor**: como camada de segurança,
   o número máximo de iterações foi limitado a 3, garantindo que mesmo em casos
   de comportamento inesperado o agent encerre a execução sem entrar em loop infinito.
   Também foi adicionado `handle_parsing_errors=True` para tratar erros de parsing
   de forma silenciosa, evitando quebras na execução.

Após essas correções, o agent passou a consultar a ferramenta exatamente uma vez
por pergunta e a gerar respostas precisas e bem fundamentadas na lei.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

# Prompt do agent
prompt = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente jurídico especializado na Lei Geral de Proteção de Dados (LGPD) do Brasil.

Seu papel é responder perguntas sobre a LGPD de forma clara, precisa e acessível.

Diretrizes:
- Use a ferramenta consultar_lgpd UMA ÚNICA VEZ para buscar informações
- Após receber o resultado da ferramenta, formule sua resposta final imediatamente
- Baseie suas respostas exclusivamente no conteúdo da LGPD
- Cite o artigo relevante quando possível (ex: "conforme o Art. 5º da LGPD...")
- Se a informação não estiver na lei, diga claramente que não encontrou
- Use linguagem simples, mas juridicamente precisa
- Nunca invente informações ou artigos"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Cria o agent com limite de iterações
agent = create_tool_calling_agent(llm, [tool], prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[tool],
    verbose=True,
    max_iterations=3,
    handle_parsing_errors=True
)

print("Agent recriado com sucesso! ")

Agent recriado com sucesso! 


In [ ]:
def perguntar(pergunta):
    resposta = agent_executor.invoke({"input": pergunta})
    print("\n Pergunta:", pergunta)
    print(" Resposta:", resposta["output"])
    print("-" * 60)

# Teste 1
perguntar("O que é a LGPD e qual seu objetivo principal?")



> Entering new AgentExecutor chain...

Invoking: `consultar_lgpd` with `{'query': 'LGPD objetivo principal'}`


Este livro traz ao leitor a Lei Geral de Proteção de Dados Pessoais (LGPD), Lei 
no 13.709/2018. Tendo por objeto o tratamento de dados pessoais, inclusive 
nos meios digitais, a LGPD visa a proteger os direitos fundamentais de liberdade 
e de privacidade e o livre desenvolvimento da personalidade da pessoa natural.

43Lei no 13.709/2018
assegurados os fundamentos, os princípios e os direitos dos titulares pre-
vistos no art. 170 da Constituição Federal e nesta Lei.
§ 2o Os regulamentos e as normas editados pela ANPD devem ser 
precedidos de consulta e audiência públicas, bem como de análises de 
impacto regulatório.
§ 3o A ANPD e os órgãos e entidades públicos responsáveis pela regula-
ção de setores específicos da atividade econômica e governamental devem 
coordenar suas atividades, nas correspondentes esferas de atuação, com 
vistas a assegurar o cumprimento de suas atri

In [ ]:
# Teste 2
perguntar("Quais são os direitos do titular dos dados segundo a LGPD?")

# Teste 3
perguntar("O que é considerado dado sensível pela LGPD?")

# Teste 4
perguntar("Quais são as penalidades para quem descumprir a LGPD?")

# Teste com algo que não tem lá
perguntar("O que a LGPD muda para os times brasileiros que viraram SAF?")

# Teste com algo que não tem lá
perguntar("Qual o país mais rico do mundo segundo a LGPD?")



> Entering new AgentExecutor chain...

Invoking: `consultar_lgpd` with `{'query': 'direitos do titular dos dados LGPD'}`


Art. 18. O titular dos dados pessoais tem direito a obter do controlador, 
em relação aos dados do titular por ele tratados, a qualquer momento e 
mediante requisição:
I – confirmação da existência de tratamento;
II – acesso aos dados;
III – correção de dados incompletos, inexatos ou desatualizados;
IV – anonimização, bloqueio ou eliminação de dados desnecessários, 
excessivos ou tratados em desconformidade com o disposto nesta Lei;
V – portabilidade dos dados a outro fornecedor de serviço ou produto, 
mediante requisição expressa, de acordo com a regulamentação da autori-
dade nacional, observados os segredos comercial e industrial;
VI – eliminação dos dados pessoais tratados com o consentimento do 
titular, exceto nas hipóteses previstas no art. 16 desta Lei;
VII – informação das entidades públicas e privadas com as quais o 
controlador realizou uso compartilha